# Explainable ML

A trained, tuned model is only useful if you can trust *why* it predicts what it
does. Explainability (XAI) comes in two flavours:

- **Global** — which features matter to the model *overall*?
- **Local** — why did it make *this* prediction for *this* input?

This mirrors the SHAP/LIME framing Python users know. We'll do a global method
(permutation importance, by hand), a local method (**Kernel SHAP** via
[`shap-rs`](https://crates.io/crates/shap-rs)) that also aggregates into a global
view, then peek at the game theory underneath.

```{note}
To keep the techniques checkable, we explain a **model whose true behaviour we
know**: feature 0 has weight 3, feature 1 weight 1, feature 2 weight 0 (pure
noise). A good explainer should recover exactly that ranking. In practice the
`predict1` below would wrap a trained model from an earlier chapter.
```

In [ ]:
:dep ndarray = { version = "0.16" }
:dep shap-rs = { version = "0.1.0" }
:dep shapley = { version = "0.1" }
use ndarray::{array, Array2, ArrayView2};
use shap_rs::{explainers::KernelExplainer, Background, Explainer, FnModel};

// The model: prediction = 3*f0 + 1*f1 + 0*f2. Feature 2 is irrelevant.
fn predict1(x: &[f64]) -> f64 { 3.0 * x[0] + 1.0 * x[1] + 0.0 * x[2] }

// A handful of samples to explain (they also serve as the SHAP background).
let data: Vec<Vec<f64>> = vec![
    vec![1.0, 2.0, 5.0], vec![2.0, 1.0, 3.0], vec![0.5, 3.0, 1.0],
    vec![3.0, 0.5, 4.0], vec![1.5, 2.5, 2.0], vec![2.5, 1.5, 0.0],
];
// The same data as an ndarray matrix — shap-rs is batch- and `ndarray`-oriented.
let x_all: Array2<f64> = Array2::from_shape_vec((data.len(), 3), data.iter().flatten().cloned().collect()).unwrap();
println!("model + {} samples ready", data.len());

## Global: permutation importance (by hand)

The idea is model-agnostic and only a few lines: **scramble one feature and see
how much the predictions move.** A feature that matters a lot will, when
shuffled, change predictions a lot; an irrelevant feature won't. (We permute a
column by reversing it, so the result is reproducible.)

```{note}
Tree/forest models in `smartcore` also expose a *built-in* feature importance,
which is the cheapest option when you're using them — but permutation importance
works for **any** model, which is why we build it here.
```

In [ ]:
{
    let base_preds: Vec<f64> = data.iter().map(|r| predict1(r)).collect();
    let n_features = 3;
    for j in 0..n_features {
        // Reverse column j across the samples = a reproducible permutation.
        let permuted: Vec<f64> = data.iter().rev().map(|r| r[j]).collect();
        let mut total_change = 0.0;
        for (i, row) in data.iter().enumerate() {
            let mut r = row.clone();
            r[j] = permuted[i];
            total_change += (predict1(&r) - base_preds[i]).abs();
        }
        println!("feature {} importance = {:.3}", j, total_change / data.len() as f64);
    }
}

Feature 0 ranks highest, feature 1 lower, feature 2 ~0 — the technique recovered
the true importance ordering.

## Local: Kernel SHAP

Permutation importance is global. To explain a *single* prediction, **SHAP**
attributes it across the features so that `base_value + sum(shap) = prediction`.
We use [`shap-rs`](https://crates.io/crates/shap-rs)'s `KernelExplainer` — a
native, model-agnostic SHAP implementation. It works on a **batch** model (an
`ndarray` matrix in, predictions out), so we wrap our `predict1` in `FnModel` and
give it the sample data as the background distribution:

In [ ]:
{
    // Wrap the same `predict1` in shap-rs's batch model trait (one row at a time).
    let model = FnModel::new(|x: ArrayView2<f64>| {
        let out: Vec<f64> = x.rows().into_iter().map(|r| predict1(r.as_slice().unwrap())).collect();
        Ok(Array2::from_shape_vec((x.nrows(), 1), out).unwrap())
    });

    // Explain one instance whose f0 is well above the background mean.
    let instance = array![[3.0_f64, 0.5, 4.0]];
    let exp = KernelExplainer::new(model, Background::new(x_all.clone()).unwrap())
        .with_nsamples(200)
        .with_seed(0)
        .explain(instance.view())
        .unwrap();

    let shap = exp.values();  // Array3: (sample, feature, output)
    println!("explaining {:?}", instance.row(0));
    for j in 0..3 {
        println!("  feature {} contributes {:+.3}", j, shap[[0, j, 0]]);
    }
    println!("base + sum(shap) = {:.3}  (model says {:.3})",
             exp.reconstructed()[[0, 0]], predict1(&[3.0, 0.5, 4.0]));
}

Feature 0 carries the largest attribution and feature 2 essentially none — and
the contributions sum back to the prediction, SHAP's defining property.

## Global importance, from SHAP

Run the same explainer over the *whole* dataset and the per-feature SHAP
magnitudes average into a **global** importance ranking — one method gives you
both the local and the global view. `shap-rs` also ships plot-ready output; here
its native, dependency-free `global_bar` SVG, rendered straight into the notebook:

In [ ]:
{
    use shap_rs::plot::svg::{global_bar, SvgOptions};
    let model = FnModel::new(|x: ArrayView2<f64>| {
        let out: Vec<f64> = x.rows().into_iter().map(|r| predict1(r.as_slice().unwrap())).collect();
        Ok(Array2::from_shape_vec((x.nrows(), 1), out).unwrap())
    });
    // Explain every sample; global_bar aggregates the mean |SHAP| per feature.
    let exp = KernelExplainer::new(model, Background::new(x_all.clone()).unwrap())
        .with_nsamples(200).with_seed(0)
        .explain(x_all.view()).unwrap();
    let svg = global_bar(&exp, &SvgOptions::default()).unwrap();
    // evcxr renders raw MIME content emitted between these markers.
    println!("EVCXR_BEGIN_CONTENT text/html\n{}\nEVCXR_END_CONTENT", svg);
}

## Where SHAP comes from: Shapley values

SHAP borrows the **Shapley value** from cooperative game theory: a fair way to
split a payout among players based on their marginal contributions to every
coalition — exactly what `shap-rs` does internally, with the model's *features*
as the players. The [`shapley`](https://docs.rs/shapley) crate shows the raw idea
on a toy game — the same math SHAP applies to features:

In [ ]:
use std::collections::HashMap;
use shapley::{Coalition, Shapley};

{
    // Two players; the value each coalition can achieve together.
    let worth = HashMap::from([
        (Coalition::new(vec![]),      0.0_f64),
        (Coalition::new(vec![1]),    10.0),
        (Coalition::new(vec![2]),    20.0),
        (Coalition::new(vec![1, 2]), 30.0),
    ]);
    let game = Shapley::new(vec![1_u64, 2], worth);
    println!("player 1 fair share = {:.1}", game.shapley_value(1).unwrap());
    println!("player 2 fair share = {:.1}", game.shapley_value(2).unwrap());
}

```{note}
**Ecosystem maturity.** Rust's explainability tooling is younger than Python's
SHAP/LIME, but [`shap-rs`](https://crates.io/crates/shap-rs) is a genuine native
SHAP library — exact / Kernel / linear / TreeSHAP, interaction values, and
plot-ready SVG output — not just a single-algorithm sketch. It's still newer and
single-author, so pin the version and re-check before relying on it. We kept
**permutation importance** by hand as the dependency-free, model-agnostic
baseline; the `shapley` toy game above is a separate early-stage crate used only
to illustrate the underlying math.
```

Next: [Persistence & Deployment](../08-persistence-deployment/saving-and-loading-models.ipynb) —
saving your trained, tuned, explained model and serving predictions from it.